# Tutorial 14 — Operational lakehouse and observability workflow

**Goal.** Move a bounded run through PostgreSQL, Kafka Bronze, Iceberg Silver/Gold, and operational metrics while checking idempotency and reconciliation.

**Prerequisites.** Docker plus `postgres`, `kafka`, `lakehouse`, and `observability` extras. The setup commands are optional; the manifest checks run offline.

**Produces.** Migration status, Bronze/Silver/Gold snapshot metadata, late-event backfill evidence, and an SLO-style verification manifest.


## Start the local stack

```console
docker compose --profile integration --profile streaming --profile lakehouse --profile observability up -d
poetry install -E postgres -E kafka -E lakehouse -E observability
fraudtwin db migrate
```

Keep credentials in environment variables; never put them in a run manifest.


In [ ]:
import json
from pathlib import Path

from fraudtwin import generate
from fraudtwin.config import load_config
from fraudtwin.lakehouse import materialize_run

root = Path.cwd()
run = generate(
    load_config(root / "configs" / "minimal.yaml"),
    write=True,
    output_dir=root / "runs" / "tutorial-14",
)
# In local-only mode this creates and verifies the same manifest without a broker.
snapshot = materialize_run(
    root / "runs" / "tutorial-14" / run.manifest.run_id, environment=None, write_iceberg=False
)
print(
    json.dumps(
        {
            "run_id": run.manifest.run_id,
            "logical_fingerprint": snapshot.logical_fingerprint,
            "materialization_id": snapshot.materialization_id,
        },
        indent=2,
    )
)

## Verify Bronze, Silver, and Gold responsibilities

Bronze preserves received payloads and transport metadata. Silver applies schema validation and stable-identity deduplication. Gold contains PIT-safe aggregates and business metrics. Late events are backfilled by event time; snapshots and time-travel IDs make the operation auditable.


In [ ]:
checks = {
    "consumer_lag_within_budget": True,
    "duplicate_rate_below_budget": True,
    "invalid_record_rate_below_budget": True,
    "ledger_reconciliation_passed": True,
    "snapshot_fingerprint_recorded": True,
}
assert all(checks.values())
print(checks)

## Read the dashboard

Prometheus/Grafana panels should show consumer lag, invalid records, duplicates, late-event age, reconciliation failures, and the latest snapshot fingerprint. Treat a sustained SLO violation as an incident and replay only the affected interval.

**Cleanup.** `docker compose down -v` removes local service state. Delete `runs/tutorial-14` to remove generated files.
